<a href="https://colab.research.google.com/github/shahwaiz-9/Deep-Learning/blob/main/Modelling_via_Word2Vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
import os

# List the files in the downloaded dataset directory
print(os.listdir(path))

['IMDB Dataset.csv']


In [ ]:
import pandas as pd

data = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))
df = data.sample(10000)
# Display the first 5 rows of the DataFrame
df.head()

,review,sentiment
11390,"Admittedly, I didn't have high expectations of...",negative
16574,"May the saints preserve us, because this movie...",negative
18806,There's simply no redeeming quality about this...,negative
21918,A THIEF IN THE NIGHT is an excellent fictional...,positive
27139,Well it's been a long year and I'm down to rev...,positive


In [ ]:
df.shape

(10000, 2)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 11390 to 936
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     10000 non-null  object
 1   sentiment  10000 non-null  object
dtypes: object(2)
memory usage: 234.4+ KB


In [ ]:
df.duplicated().sum()

np.int64(9)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
# Checking balance of classes

df['sentiment'].value_counts()


,count
sentiment,
negative,4996
positive,4995


In [ ]:
pip install contractions

In [ ]:
import re
import nltk
import contractions
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

In [ ]:

# Download once (if not already done)
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
stopwords = nltk.corpus.stopwords.words('english')
lemmatizer = WordNetLemmatizer()


In [ ]:
df['review'].iloc[0]

'Admittedly, I didn\'t have high expectations of "Corky Romano." But then again, who did? However, I felt it deserved the benefit of the doubt. I had no high hopes of "Joe Dirt" either--another recent comedy starring an SNL cast member--and I ended up being pleasantly surprised. But this film is just as bad as it looks in the previews. Chris Kattan is actually a talented comic actor--contrary to what you might think after watching this movie--with great energy. He\'s been in many hilarious SNL skits, and I think he\'s one of the most talented cast members on SNL as of now. In this case, he\'s given a lame, pointless script and he tries to remedy each scene with his incessant mugging. Throughout each scene, he attempts a lame Jerry Lewis act and fails miserably. Jerry Lewis knew how to pull off this type of physical comedy, not to mention he worked with much better writing. Kattan simply looks like some ignorant fool with ADHD who had one too many Cafe Lattes. He doesn\'t even wait for 

In [ ]:
def clean_text(text):

  text = re.sub(r'<.*?>', '', text)
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = contractions.fix(text)
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)

  return text


In [ ]:
df['review'] = df['review'].apply(clean_text)

In [ ]:
df['review'].iloc[0]

'admittedly i did not have high expectations of corky romano but then again who did however i felt it deserved the benefit of the doubt i had no high hopes of joe dirt eitheranother recent comedy starring an snl cast memberand i ended up being pleasantly surprised but this film is just as bad as it looks in the previews chris kattan is actually a talented comic actorcontrary to what you might think after watching this moviewith great energy he is been in many hilarious snl skits and i think he is one of the most talented cast members on snl as of now in this case he is given a lame pointless script and he tries to remedy each scene with his incessant mugging throughout each scene he attempts a lame jerry lewis act and fails miserably jerry lewis knew how to pull off this type of physical comedy not to mention he worked with much better writing kattan simply looks like some ignorant fool with adhd who had one too many cafe lattes he does not even wait for the punchline he assumes we wil

In [ ]:
def get_wordnet_pos(tag):
    """Convert NLTK POS tag to WordNet POS tag"""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default fallback

In [ ]:
def tokenize_lemmitize(text):

  # Word tokenization
  tokens = word_tokenize(text)

  pos_tags = nltk.pos_tag(tokens)

  lemmatized = []

  for word, tag in pos_tags:
    wn_tag = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word, pos=wn_tag)

    if word not in stopwords and len(lemma) > 1:
      lemmatized.append(lemma)

  return lemmatized


In [ ]:
df['processed_review'] = df['review'].apply(tokenize_lemmitize)

In [ ]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

**Feature Extraction**

In [ ]:
pip install gensim

**Modeling**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from gensim.models import Word2Vec

In [ ]:
model = Word2Vec(
    sentences=df['processed_review'],
    window=5,
    min_count=2,
    max_vocab_size=50000,
    vector_size=100,
    workers=4,
    sg=1
)

In [ ]:

# Parameters
max_words = 10000
max_len = 100

# Initialize and fit tokenizer
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['processed_review'])

# Convert tokens to sequences of integers
sequences = tokenizer.texts_to_sequences(df['processed_review'])

# Pad sequences to ensure uniform input size
X = pad_sequences(sequences, maxlen=max_len)
y = df['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
vector_size = model.vector_size # Should be 100 based on your config

# 2. Create the matrix
embedding_matrix = np.zeros((vocab_size, vector_size))
for word, i in word_index.items():
    if word in model.wv:
        embedding_matrix[i] = model.wv[word]

In [ ]:
print(f"Vocabulary size: {len(model.wv.key_to_index)}")
print(f"Total words in corpus: {model.corpus_total_words}")

Vocabulary size: 20680
Total words in corpus: 1176623


Let's check the vector representation for a word and its dimensionality. We can pick a common word from our processed reviews, for example, 'love'.

In [ ]:
word_vector = model.wv['love']
print("Vector for 'hate':", word_vector[:5]) # Print first 5 dimensions
print("Shape of the vector for 'love':", word_vector.shape)

Vector for 'hate': [-0.15574318 -0.23975118 -0.05538243 -0.33563867 -0.52610064]
Shape of the vector for 'love': (100,)


Let's check the quality of our Word2Vec embeddings by finding words similar to a common word like 'good' and 'bad'.

In [ ]:
print("Words similar to 'good':")
if 'good' in model.wv:
    for word, similarity in model.wv.most_similar('good'):
        print(f"  {word}: {similarity:.4f}")
else:
    print("  'good' not in vocabulary.")

print("\nWords similar to 'bad':")
if 'bad' in model.wv:
    for word, similarity in model.wv.most_similar('bad'):
        print(f"  {word}: {similarity:.4f}")
else:
    print("  'bad' not in vocabulary.")

Words similar to 'good':
  eh: 0.7638
  decent: 0.7634
  darn: 0.7623
  goodbut: 0.7620
  impressed: 0.7589
  goodi: 0.7564
  onthe: 0.7563
  bearable: 0.7536
  competently: 0.7518
  passable: 0.7514

Words similar to 'bad':
  worse: 0.7755
  worst: 0.7744
  stink: 0.7607
  awful: 0.7575
  soo: 0.7506
  abysmal: 0.7472
  lousy: 0.7435
  atrocious: 0.7417
  uwe: 0.7405
  seenthe: 0.7370


In [ ]:
y = df['label'].values

In [ ]:
import numpy as np
review_lengths = [len(tokens) for tokens in df['processed_review']]
max_len = int(np.percentile(review_lengths, 99))
print(f"99% of documents have a length less than or equal to: {max_len} tokens")

99% of documents have a length less than or equal to: 455 tokens


In [ ]:
max_document_length = max(review_lengths)
print(f"The maximum document length is: {max_document_length} tokens")

The maximum document length is: 1387 tokens


In [ ]:
num_documents_last_1_percent = sum(1 for length in review_lengths if length > max_len)
print(f"Number of documents in the last 1% (longer than {max_len} tokens): {num_documents_last_1_percent}")
print(f"This represents {num_documents_last_1_percent / len(review_lengths) * 100:.2f}% of the total documents.")

Number of documents in the last 1% (longer than 455 tokens): 100
This represents 1.00% of the total documents.


In [ ]:
review_lengths = [len(tokens) for tokens in df['processed_review']]
df = df[np.array(review_lengths) <= max_len]
print(f"New DataFrame shape after removing long documents: {df.shape}")

New DataFrame shape after removing long documents: (9891, 4)


In [ ]:
y = df['label'].values

See 99 documents are causing double dimensions complexity so in this case it is good to drop them and continue with remianing data

In [ ]:
length = max_len # Aligning sequence length with padding length

In [ ]:
# X.shape

In [ ]:
class DataGenerator(tf.keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, reviews, labels, embedding_matrix, sequence_length, embedding_dim, batch_size=32, shuffle=True):
        self.reviews = reviews # These are now integer sequences (X_train or X_test)
        self.labels = labels
        self.embedding_matrix = embedding_matrix # Added embedding_matrix
        self.sequence_length = sequence_length
        self.embedding_dim = embedding_dim
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        'Denotes the number of batches per epoch'
        return int(np.floor(len(self.reviews) / self.batch_size))

    def __getitem__(self, index):
        'Generate one batch of data'
        # Generate indexes of the batch
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        # Find list of IDs and get the corresponding integer sequences and labels
        batch_review_indices = [self.reviews[k] for k in indexes]
        batch_labels = [self.labels[k] for k in indexes]

        # Generate data (embedding vectors)
        X, y = self.__data_generation(batch_review_indices, batch_labels)
        return X, y

    def on_epoch_end(self):
        'Updates indexes after each epoch'
        self.indexes = np.arange(len(self.reviews))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_review_indices, batch_labels):
        'Generates data containing batch_size samples (embedding vectors)'
        # X : (batch_size, sequence_length, embedding_dim)
        # y : (batch_size,)
        X = np.zeros((self.batch_size, self.sequence_length, self.embedding_dim))
        y = np.array(batch_labels)

        for i, seq_of_indices in enumerate(batch_review_indices):
            # Map each integer index to its corresponding word embedding vector
            # seq_of_indices is already padded with 0s (for padding/unknown words)
            # embedding_matrix[0] should be a zero vector, ensuring padding works correctly.
            X[i,] = self.embedding_matrix[seq_of_indices]

        return X, y

In [ ]:
EMBEDDING_DIM = model.vector_size
batch_size = 32
train_generator = DataGenerator(
    X_train,
    y_train,
    embedding_matrix=embedding_matrix, # Pass embedding_matrix
    sequence_length=length,
    embedding_dim=EMBEDDING_DIM,
    batch_size=batch_size,
    shuffle=True
)

val_generator = DataGenerator(
    X_test,
    y_test,
    embedding_matrix=embedding_matrix, # Pass embedding_matrix
    sequence_length=length,
    embedding_dim=EMBEDDING_DIM,
    batch_size=batch_size,
    shuffle=False # No need to shuffle validation data
)

In [ ]:
lstm_model = Sequential([

    LSTM(
        128,
        return_sequences=True,
    ),
    LSTM(
        64,
        return_sequences=False,
    ),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])




In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",  # or "categorical_crossentropy"
    metrics=["accuracy"]
)

In [ ]:
lstm_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_11 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = lstm_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20, # Increased epochs from 10 to 20
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)

Epoch 1/20


ValueError: Input 0 with name 'None' of layer 'lstm_8' is incompatible with the layer: expected ndim=3, found ndim=4. Full shape received: (None, 449, 100, 100)